# Module 09 · Unit 03
# Regular Expressions and ReDoS

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Adversarial Enumeration \[AE\] · Protocol Verification \[PV\] |
| **Refresher** | Module 09 · Unit 01 — Finite Automata (NFA structure) |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain how a regular expression compiles to an NFA via **Thompson's construction**
2. Describe the difference between **DFA-based** and **backtracking NFA** regex engines
3. Identify the **structural patterns** that cause catastrophic backtracking
4. Formally characterise a ReDoS-vulnerable pattern and the input that triggers it
5. Measure the exponential time growth empirically using the `re` module
6. Apply safe-regex principles: rewrite patterns to eliminate ambiguous repetition


## 🔣 Symbol Primer

This unit uses the NFA notation from Unit 01 plus standard regex operator symbols.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $r^*$ | Kleene star | "$r$ star" | Zero or more repetitions of $r$ |
| $r^+$ | Kleene plus | "$r$ plus" | One or more repetitions of $r$ |
| $r?$ | Optional | "$r$ optional" | Zero or one occurrence of $r$ |
| $r_1 \mid r_2$ | Alternation | "$r_1$ or $r_2$" | Either $r_1$ or $r_2$ |
| $r_1 r_2$ | Concatenation | "$r_1$ then $r_2$" | $r_1$ followed by $r_2$ |
| $[abc]$ | Character class | "any of a, b, c" | One character matching any in the set |
| $\varepsilon$ | Epsilon transition | "epsilon" | NFA transition consuming no input |

> **The key connection.** Every regular expression describes a language that
> can be recognised by an NFA — and the NFA can be constructed mechanically
> from the regex structure (Thompson's construction). The NFA can then be
> simulated in two ways: using DFA-based matching (linear time, more memory)
> or backtracking NFA simulation (potentially exponential time, less memory).
> ReDoS exploits the worst case of backtracking simulation.

---


## 1 · From Regular Expression to NFA — Thompson's Construction

**Thompson's construction** converts any regular expression to an NFA with
$O(|r|)$ states in $O(|r|)$ time, where $|r|$ is the length of the regex.
The construction is compositional: each regex operator produces an NFA fragment
from its sub-expression NFAs.

### Building Blocks

**Single character $a$:** two states, one transition on $a$.

$$q_0 \xrightarrow{a} q_1 \qquad q_1 \in F$$

**Concatenation $r_1 r_2$:** connect the accept state of $r_1$ to the start
state of $r_2$ with an $\varepsilon$-transition.

**Alternation $r_1 \mid r_2$:** add a new start state with $\varepsilon$-transitions
to the start states of both $r_1$ and $r_2$; add a new accept state reached
by $\varepsilon$-transitions from both accept states.

**Kleene star $r^*$:** add a new start/accept state with $\varepsilon$-transitions
to the start and from the accept state of $r$, plus an $\varepsilon$-transition
from $r$'s accept state back to its start state (the loop).

### Why the Loop Creates ReDoS Risk

The $\varepsilon$-loop in the Kleene star is the structural feature that enables
backtracking to become exponential. Consider $(a^+)^+$:

1. The outer $^+$ creates a loop in the NFA
2. The inner $a^+$ creates another loop
3. Both loops can be entered and exited in multiple ways for many inputs
4. A backtracking engine explores **all combinations** of how the loops partition
   the input before concluding "no match"

For an input string of $n$ `a` characters followed by a non-matching character,
the number of ways to partition the `a` characters into groups is exponential
in $n$ — the backtracking engine tries them all.


## 2 · Two Regex Engine Strategies

### Strategy 1 — Backtracking NFA Simulation

Most scripting languages (Python `re`, JavaScript, Java, PHP, Ruby) use a
backtracking NFA engine:

1. Build the NFA from the regex (Thompson's construction)
2. Simulate the NFA on the input using **depth-first search with backtracking**
3. When a non-match is found, backtrack and try the next alternative

**Time complexity:** $O(2^n)$ worst case for $n$-character input on vulnerable patterns.

**Space complexity:** $O(n)$ — only one active path explored at a time.

### Strategy 2 — DFA-Based Matching (RE2, Rust `regex`)

1. Convert the NFA to a DFA via subset construction (Unit 01)
2. Run the DFA deterministically — one state per input character

**Time complexity:** $O(n)$ — guaranteed linear time on input length.

**Space complexity:** $O(2^m)$ worst case where $m$ is the number of NFA states —
the state explosion from subset construction happens at compile time, not match time.

### The Security Tradeoff

| Engine | Match time | Compile time | Supports backreferences? | ReDoS-safe? |
|---|---|---|---|---|
| Backtracking NFA | $O(2^n)$ worst | $O(\|r\|)$ | ✓ | ✗ |
| DFA (RE2, Rust regex) | $O(n)$ | $O(2^{\|r\|})$ worst | ✗ | ✓ |

**Backreferences** (e.g., `(\w+) \1` matching repeated words) require backtracking
and cannot be expressed as DFAs. If your regex uses backreferences, a backtracking
engine is unavoidable — but you must audit carefully for catastrophic patterns.


## 3 · ReDoS — Structural Characterisation

A **ReDoS-vulnerable pattern** has three components:

1. **Ambiguous repetition** — two or more ways to match the same prefix of the input
2. **Common suffix requirement** — after the repetition, a character or pattern is required
3. **Suffix mismatch** — the attacker-controlled input satisfies the repetition
   but not the suffix

The canonical form is:

$$\underbrace{(a^+)^+}_{\text{ambiguous repetition}} \underbrace{b}_{\text{suffix}}$$

On input $\underbrace{aaa\cdots a}_{n} c$ (n `a`s followed by a non-`b` character):
- The engine tries to match the `a`s against $(a^+)^+$ — succeeds multiple ways
- Then fails to match `c` against `b`
- Backtracks and tries a different partition of the `a`s — fails again
- Continues until all $2^{n-1}$ partitions are exhausted

**Number of backtracks:** $\Theta(2^n)$ — exponential in the input length.

### Common Vulnerable Patterns in the Wild

| Pattern | Vulnerable because |
|---|---|
| `(a+)+` | Nested repetition on same character class |
| `([a-z]+)*` | Star over plus |
| `(a\|aa)+` | Alternation with overlapping matches |
| `(\d+\.)+\d+` | Used for IP-like validation — the dots force backtracking |
| `^(([a-z])+\.)+[A-Z]([a-z])+$` | Email-like validator — multiple overlapping groups |

The last two are particularly dangerous because they appear in legitimate
email and URL validation code — and they are commonly copy-pasted from
Stack Overflow without ReDoS analysis.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AE\] \[PV\]

| ReDoS concept | Security meaning |
|---|---|
| **Ambiguous NFA** | Multiple paths through the regex — backtracking engine tries all |
| **Backtracking engine** | $O(2^n)$ worst case — attacker controls $n$ |
| **Crafted input** | String maximising backtracking — easy to construct once pattern is known |
| **Denial of service** | Server thread blocked for seconds/minutes on one regex match |
| **AI API impact** | Input validation on LLM API endpoints often uses regex; ReDoS = blocked inference |
| **DFA engine** | $O(n)$ guarantee — immune to ReDoS at cost of no backreferences |
| **Static analysis** | Tools like `safe-regex`, `vuln-regex-detector` identify vulnerable patterns |

**The AI-specific risk.** LLM APIs and MCP servers commonly validate inputs
using regular expressions — checking that tool call parameters match expected
formats (email addresses, API keys, structured IDs). A single ReDoS-vulnerable
pattern in a hot path can be triggered by any user input, blocking the validation
thread and degrading the entire inference pipeline. Unlike most DoS vulnerabilities,
ReDoS requires no special privileges — any user who can submit input can trigger it.

---


## Python: Visualization & Verification

**1 · Thompson's Construction** — implement the NFA construction for the three
core operators (concatenation, alternation, Kleene star) and visualise the
resulting NFA for a simple vulnerable pattern.

**2 · ReDoS Timing Demonstration** — measure match time for a vulnerable pattern
on inputs of increasing length; plot the exponential growth and compare against
a safe equivalent; demonstrate that the vulnerability is real and measurable.

**3 · Vulnerability Classifier** — implement heuristic checks for the three
structural ReDoS patterns; classify a set of real-world regex patterns; for each
vulnerable pattern, suggest a safe rewrite.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import re, time

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Thompson's Construction — NFA from Regex

We implement a simplified Thompson's construction for the three core operators
and visualise the NFA for the pattern `(a|b)*abb` — the classic textbook example
that accepts all strings ending in "abb". This is a benign pattern. We then
show how adding nested repetition creates the ambiguity that enables ReDoS.


In [ ]:
# ── 1 · Thompson's Construction — Simplified Visualization ───────────────────

class NFAFragment:
    """An NFA fragment with a single start and accept state."""
    def __init__(self):
        self.states      = set()
        self.transitions = {}  # {(state, symbol): set of next states}
        self.start       = None
        self.accept      = None
        self._counter    = [0]

    def new_state(self, label=None):
        sid = self._counter[0]
        self._counter[0] += 1
        name = label if label else f"q{sid}"
        self.states.add(name)
        return name

    def add_transition(self, src, sym, dst):
        key = (src, sym)
        if key not in self.transitions:
            self.transitions[key] = set()
        self.transitions[key].add(dst)

# Build NFA for (a|b)*abb manually for clear visualisation
# States: q0,q1,q2,q3,q4,q5,q6,q7,q8,q9,q10
# q0 = start, q10 = accept
# (a|b)* part: q0 --ε--> q1, q1 --a--> q2, q1 --b--> q4,
#              q2 --ε--> q3, q4 --ε--> q5, q3 --ε--> q6, q5 --ε--> q6
#              q6 --ε--> q1 (loop), q6 --ε--> q7 (continue to abb)
# abb part:    q7 --a--> q8 --b--> q9 --b--> q10

states = list(range(11))
state_labels = {i: f"q{i}" for i in states}

transitions = {
    (0, 'ε'): {1, 7},          # start: either (a|b)* loop or go to 'a'
    (1, 'ε'): {2, 4},          # choice: a or b
    (2, 'a'): {3},
    (4, 'b'): {5},
    (3, 'ε'): {6},
    (5, 'ε'): {6},
    (6, 'ε'): {1, 7},          # loop back or continue
    (7, 'a'): {8},
    (8, 'b'): {9},
    (9, 'b'): {10},
}
start_state  = 0
accept_state = 10

# Build networkx graph for visualisation
G = nx.MultiDiGraph()
G.add_nodes_from(states)

edge_info = {}  # (src,dst) -> labels
for (src, sym), dsts in transitions.items():
    for dst in dsts:
        key = (src, dst)
        edge_info.setdefault(key, []).append('ε' if sym == 'ε' else sym)
        if not G.has_edge(src, dst):
            G.add_edge(src, dst)

fig, ax = plt.subplots(figsize=(13, 5))
pos = {
    0: (0,0), 1: (1.5,0), 2: (2.5, 0.8), 3: (3.5, 0.8),
    4: (2.5,-0.8), 5: (3.5,-0.8), 6: (4.5, 0), 7: (5.5, 0),
    8: (7, 0), 9: (8.5, 0), 10: (10, 0),
}

nc = [TS_GREEN  if n == accept_state else
      TS_AMBER  if n == start_state  else TS_BLUE
      for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, ax=ax, node_color=nc, node_size=700, alpha=0.92)
nx.draw_networkx_labels(G, pos, ax=ax,
                        labels={n: f"q{n}" for n in G.nodes()},
                        font_color='white', font_size=8, font_weight='bold')

for (src, dst), syms in edge_info.items():
    label = ','.join(syms)
    rad   = 0.35 if src > dst else (0.2 if (src,dst) in [(6,1)] else 0.0)
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=[(src, dst)],
                           edge_color=TS_AMBER if 'ε' in syms else TS_BLUE,
                           arrows=True, arrowsize=16, width=1.8,
                           connectionstyle=f'arc3,rad={rad}', alpha=0.8)
    mx = (pos[src][0] + pos[dst][0]) / 2 + (0 if rad == 0 else 0.1)
    my = (pos[src][1] + pos[dst][1]) / 2 + (0.25 if rad == 0 else 0.4)
    ax.text(mx, my, label, ha='center', fontsize=8,
            color=TS_AMBER if 'ε' in syms else TS_BLUE, fontweight='bold')

ax.annotate('', xy=(pos[0][0]-0.1, pos[0][1]),
            xytext=(pos[0][0]-0.7, pos[0][1]),
            arrowprops=dict(arrowstyle='->', color=TS_GRAY, lw=2))

legend = [
    mpatches.Patch(color=TS_AMBER, label='Start state / ε-transitions'),
    mpatches.Patch(color=TS_BLUE,  label='Regular states / symbol transitions'),
    mpatches.Patch(color=TS_GREEN, label='Accept state'),
]
ax.legend(handles=legend, loc='upper left', fontsize=8)
ax.set_title("NFA for (a|b)*abb — Thompson's Construction
"
             "ε-transitions (amber) allow non-deterministic jumps; "
             "the q6→q1 back-edge creates the Kleene star loop",
             pad=10, fontsize=10)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()

print("NFA structure summary:")
print(f"  States: q0 (start) through q10 (accept)")
print(f"  ε-transitions (amber): enable non-determinism — no input consumed")
print(f"  Back-edge q6→q1: implements the Kleene star loop in (a|b)*")
print(f"  The NFA has multiple active paths simultaneously for each input character.")
print(f"  A backtracking engine explores these paths one at a time → exponential for bad patterns.")


**What do you see?**

The NFA diagram shows the full structure of `(a|b)*abb`. The amber epsilon transitions
($\varepsilon$) are the non-deterministic jumps — they allow the NFA to be in
multiple states simultaneously without consuming any input.

The critical back-edge $q_6 \to q_1$ implements the Kleene star loop: after
matching one $a$ or $b$, the NFA can loop back and match another. This is what
enables `(a|b)*` to match any string of a's and b's.

For this specific pattern `(a|b)*abb`, there is no ReDoS risk — the non-terminal
`abb` at the end means that for most inputs, the engine quickly determines whether
the string ends in "abb" and halts. The vulnerability arises when nested repetition
creates ambiguity with *no required suffix* to act as a disambiguation anchor.


### 2 · ReDoS Timing Demonstration

We measure the actual match time for three patterns on inputs of increasing
length. The vulnerable pattern `(a+)+b` shows exponential growth; a safe
alternative `a+b` shows linear growth; Python's `re` module (backtracking engine)
demonstrates the vulnerability concretely.

**Safety note:** all inputs here are bounded to keep runtimes under a few seconds.
Real ReDoS attacks use inputs of 30–50+ characters to cause multi-second or minute-long hangs.


In [ ]:
# ── 2 · ReDoS Timing Demonstration ──────────────────────────────────────────
import re, time

def measure_time(pattern, input_str, timeout=5.0):
    """Measure regex match time with a safety timeout."""
    compiled = re.compile(pattern)
    t0 = time.perf_counter()
    try:
        result = compiled.match(input_str)
        elapsed = time.perf_counter() - t0
        return elapsed, result is not None
    except Exception:
        return time.perf_counter() - t0, False

# Patterns to compare
patterns = {
    'Vulnerable: (a+)+b':    r'(a+)+b',
    'Safe equivalent: a+b':  r'a+b',
    'Vulnerable: (a|aa)+b':  r'(a|aa)+b',
}

# Attack input: n 'a' characters followed by 'c' (never matches 'b')
lengths = list(range(1, 28))

print("ReDoS Timing Comparison")
print("Input: 'a' × n + 'c'  (never matches any pattern — worst case)
")
print(f"{'n':>4}", end='')
for name in patterns:
    short = name.split(':')[0].strip()[:12]
    print(f"  {short:>16}", end='')
print()
print("─" * 65)

timing_data = {name: [] for name in patterns}

for n in lengths:
    attack_input = 'a' * n + 'c'
    print(f"{n:>4}", end='')
    for name, pat in patterns.items():
        if timing_data[name] and timing_data[name][-1] > 2.0:
            timing_data[name].append(None)
            print(f"  {'(timeout)':>16}", end='')
        else:
            t, _ = measure_time(pat, attack_input)
            timing_data[name].append(t)
            marker = '⚠' if t > 0.1 else ' '
            print(f"  {t*1000:>14.2f}ms{marker}", end='')
    print()

# ── Plot timing curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = [TS_RED, TS_GREEN, TS_AMBER]
for ax_idx, (ax, yscale) in enumerate([(axes[0], 'linear'), (axes[1], 'log')]):
    for (name, times), color in zip(timing_data.items(), colors):
        valid_ns = [n for n, t in zip(lengths, times) if t is not None]
        valid_ts = [t * 1000 for t in times if t is not None]
        label    = name.split(':')[0].strip()
        ax.plot(valid_ns, valid_ts, color=color, lw=2.5,
                marker='o', markersize=5, label=label)

    ax.set_xlabel('Input length n (number of "a" characters)')
    ax.set_ylabel('Match time (milliseconds)')
    ax.set_yscale(yscale)
    ax.set_title(f'ReDoS Timing — {yscale.capitalize()} Scale', fontweight='bold', pad=8)
    ax.legend(fontsize=9)
    ax.axhline(100, color=TS_GRAY, lw=1, linestyle='--', alpha=0.5,
               label='100ms threshold')

fig.patch.set_facecolor('white')
plt.suptitle('ReDoS Vulnerability: Exponential vs Linear Match Time
'
             "Vulnerable pattern: (a+)+b | Safe: a+b | Input: 'a'×n + 'c'",
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**What do you see?**

The timing data shows the vulnerability concretely. The safe pattern `a+b` matches
(or rejects) in constant microseconds regardless of input length — truly linear
time. The vulnerable pattern `(a+)+b` starts fast but grows exponentially: each
additional `a` character roughly doubles the match time.

On the log scale (right chart), the safe pattern is a flat line while the
vulnerable patterns climb steeply. At around $n = 25$ the vulnerable patterns
begin taking hundreds of milliseconds. At $n = 40$, a real server would stall
for many seconds on a single regex match — a complete denial of service for one
input validation call.

**The AI API threat.** An LLM inference endpoint that validates user input with
a vulnerable pattern before passing it to the model can be fully denied by any
user who knows the pattern. The user does not need elevated privileges, network
access, or special knowledge — they just need to know or guess that the input
field uses a vulnerable regex. This is a particularly low-cost attack.


### 3 · Vulnerability Classifier and Safe Rewrites

We implement heuristic checks for the three main ReDoS structural patterns and
apply them to a set of real-world regex patterns from input validation code.
For each vulnerable pattern we suggest a safe rewrite and verify the rewrite
is equivalent on a test suite.


In [ ]:
# ── 3 · Vulnerability Classifier and Safe Rewrites ───────────────────────────

def check_redos_heuristics(pattern):
    """
    Check for common ReDoS structural patterns.
    Returns list of (risk_level, description) findings.
    """
    findings = []

    # Pattern 1: Nested quantifiers on same character class (a+)+ or (a*)* etc.
    if re.search(r'\([^)]*[+*][^)]*\)[+*]', pattern):
        findings.append(('HIGH',
            'Nested quantifier: (X+)+ or (X*)* pattern — '
            'exponential backtracking on non-matching suffix'))

    # Pattern 2: Overlapping alternation inside quantifier (\w+|\w)+ etc.
    if re.search(r'\([^)]*\|[^)]*\)[+*]', pattern):
        findings.append(('HIGH',
            'Quantified alternation: (X|Y)+ where X and Y overlap — '
            'exponential alternatives to explore'))

    # Pattern 3: Multiple adjacent quantifiers without anchoring
    if re.search(r'[+*][+*]', pattern):
        findings.append(('MEDIUM',
            'Adjacent quantifiers — may cause backtracking without anchoring'))

    # Pattern 4: No anchors on long alternation
    if re.search(r'\|', pattern) and not re.search(r'^\^|\$$', pattern):
        findings.append(('LOW',
            'Unanchored alternation — consider adding ^ and $ anchors'))

    return findings

# Real-world patterns to analyse
patterns_to_check = [
    # Vulnerable patterns
    (r'^(([a-z])+\.)+[A-Z]([a-z])+$',
     'Email-like validator (copy-paste from internet)',
     r'^([a-z]+\.)+[A-Z][a-z]+$'),

    (r'^(\d+\.)+\d+$',
     'Version string validator (e.g., 1.2.3)',
     r'^\d+(\.\d+)+$'),

    (r'^(a+)+$',
     'Minimal vulnerable: nested plus',
     r'^a+$'),

    (r'^([a-zA-Z0-9]([a-zA-Z0-9\-]*[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}$',
     'Hostname validator',
     r'^[a-zA-Z0-9]([a-zA-Z0-9\-]*[a-zA-Z0-9])?(\.[a-zA-Z0-9]([a-zA-Z0-9\-]*[a-zA-Z0-9])?)*\.[a-zA-Z]{2,}$'),

    # Safe patterns
    (r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$',
     'IPv4 validator (bounded repetition)',
     None),

    (r'^[A-Za-z0-9+/]{43}=$',
     'Base64 API key (fixed length)',
     None),
]

print("ReDoS Vulnerability Analysis")
print("=" * 70)

for pattern, description, safe_rewrite in patterns_to_check:
    findings = check_redos_heuristics(pattern)
    risk     = max((f[0] for f in findings), default='NONE',
                   key=lambda r: {'NONE':0,'LOW':1,'MEDIUM':2,'HIGH':3}[r])
    icon     = {'NONE':'✓','LOW':'⚠','MEDIUM':'⚠','HIGH':'✗'}[risk]
    print(f"
{icon} [{risk}] {description}")
    print(f"  Pattern: {pattern}")
    for _, desc in findings:
        print(f"  Finding: {desc}")
    if safe_rewrite:
        print(f"  Safe rewrite: {safe_rewrite}")

        # Verify rewrite equivalence on test inputs
        test_inputs = ['abc.D', 'a.b.C', '1.2.3', 'aaa', 'a.B.c', '', 'AA.b']
        orig_compiled  = re.compile(pattern)
        safe_compiled  = re.compile(safe_rewrite)
        mismatches = [(t, bool(orig_compiled.match(t)), bool(safe_compiled.match(t)))
                      for t in test_inputs
                      if bool(orig_compiled.match(t)) != bool(safe_compiled.match(t))]
        if mismatches:
            print(f"  ⚠ Rewrite differs on: {mismatches}")
        else:
            print(f"  Equivalence check on {len(test_inputs)} test inputs: ✓")

print("
" + "=" * 70)
print("Summary:")
print("  HIGH risk patterns require immediate rewrite or DFA engine")
print("  Use bounded quantifiers {n,m} where possible instead of + or *")
print("  Anchor all patterns with ^ and $ to prevent partial matching")
print("  For production AI API validation: use re2 library or equivalent")


**What do you see?**

The classifier correctly identifies the high-risk patterns. The email-like and
hostname validators are particularly dangerous because they are widely copy-pasted
without ReDoS analysis. Both use nested or repeated groups with overlapping matches.

The safe rewrites restructure the patterns to eliminate ambiguity:

- **Avoid nesting** — `([a-z]+\.)+` instead of `(([a-z])+\.)+` removes one
  layer of the dangerous nesting
- **Use atomic groups or possessive quantifiers** where the engine supports them
- **Prefer bounded quantifiers** `{n,m}` over unbounded `+` and `*` where the
  input format permits
- **For critical paths: use a DFA engine** — the `re2` library (also available
  as Google's `re2` Python binding) guarantees linear-time matching at the cost
  of not supporting backreferences

The IPv4 and Base64 patterns at the bottom are safe: they use bounded repetition
`{n,m}` and fixed character classes with no nested alternation. These are the
patterns to model input validators on.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. NFA PATH COUNT
#    For the pattern (a+)+b and input 'aaaa...c' (n a's followed by c):
#    Implement a function count_nfa_paths(n) that counts the number of
#    distinct NFA computation paths the backtracking engine must explore
#    before concluding no match.
#    Verify: does the path count grow as O(2^n)?
#    Plot path count vs n alongside the timing data from Section 2.
#
# 2. POSSESSIVE QUANTIFIERS
#    Possessive quantifiers (a++ in some engines) prevent backtracking:
#    once a match is committed, it is never reconsidered.
#    Python's `re` module does not support possessive quantifiers, but
#    `regex` (pip install regex) does.
#    Install the `regex` module, rewrite (a+)+ as (a++)+ and measure
#    whether the ReDoS vulnerability is eliminated.
#    Explain: why does possessive quantifier prevent backtracking here?
#
# 3. LLM API VALIDATOR AUDIT
#    Consider an LLM API that validates tool call parameters using these patterns:
#      email:     r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
#      url:       r'^https?://([\w-]+\.)+[\w-]+(/[\w-./?%&=]*)?$'
#      json_key:  r'^[a-zA-Z_][a-zA-Z0-9_]*$'
#    For each: (a) check for ReDoS vulnerability, (b) craft the minimal
#    attack input, (c) measure the time at n=20 characters, (d) propose a fix.

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Thompson's construction** | Regex → NFA in $O(\|r\|)$ | Every regex has an explicit NFA structure |
| **Backtracking NFA engine** | DFS through NFA paths with backtracking | $O(2^n)$ worst case on ambiguous patterns |
| **DFA engine (RE2)** | Subset-constructed DFA | $O(n)$ guaranteed — ReDoS-immune |
| **Ambiguous repetition** | Multiple NFA paths through same input | The structural cause of ReDoS |
| **Catastrophic backtracking** | Exponential path exploration on non-match | The attack: maximise the backtrack count |
| **Crafted input** | $n$ matching chars + non-matching suffix | Triggers all $2^n$ partitions to be tried |
| **Safe rewrite** | Remove nested quantifiers; use bounded forms | Eliminates ambiguity from the NFA |

---

## Module 09 Complete

| Unit | Content | Payoff |
|---|---|---|
| **01** | DFA, NFA, subset construction, regular languages | The formal model for any bounded-state system |
| **02** | Protocol state machines, MCP handshake, violation path finder | Protocol correctness = DFA invariant; violations = paths to ERROR |
| **03** | Thompson's construction, backtracking engines, ReDoS | Regex vulnerability is NFA structure + backtracking engine choice |

The three units form one argument: **a finite automaton is the right formal model
for any system with bounded state — whether it is a protocol, a language, or a
regex engine. The security vulnerabilities that arise in each are structural
properties of the automaton, detectable through formal analysis.**

---

## Up Next

**Module 10 — Capstone**

Every formal tool from the course applied to a single agentic AI system.
The capstone assembles Boolean logic, predicate logic, set theory, graph theory,
tree analysis, combinatorics, number theory, and automata into one coherent
formal threat model — the kind of document you would present to a security
review board or include in a PhD research proposal.

$\rightarrow$ `modules/module-10/unit-01-capstone-threat-model.ipynb`
